In [ ]:
import subprocess

def run(command):
    """Run a shell command, show its output live, and fail the notebook if the command fails."""
    print("$", command, flush=True)
    process = subprocess.Popen(command, shell=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    for line in process.stdout:
        print(line, end="", flush=True)
    if process.wait() != 0:
        raise RuntimeError(f"exit code {process.returncode}: {command}")


In [ ]:
run("nvidia-smi")
run("pip install -q transformers==4.57.1 peft==0.17.1 accelerate")

In [ ]:
%%writefile convert_mapqa.py
#!/usr/bin/env python3
"""Convert MapQA-S into derenderer training files: one map image + the JSON the model should output.

Continuous legend (smooth color bar):
    {"title":"The Percentage of ceremonies in the USA","legend_title":"Value","legend_type":"continuous",
     "data_type":"percentage","values":{"AK":37.9,"AL":39.8,...,"WY":47.4}}

Discrete legend (ranges):
    {"title":"The Number of shots in the USA","legend_title":null,"legend_type":"discrete","data_type":"number",
     "ranges":["7548477-7658039","7668882-7761001","7811458-7859166","7920264-8002936"],
     "values":{"AK":0,"AL":0,...,"WY":1}}

Rules:
- values are keyed by state code, sorted, all 50 states every time
- discrete: each state gets its range number (0 = lowest range); "ranges" lists the legend ranges lowest
  first, whatever order the image shows them in; states with no data get null (the image shows "N/A")
- continuous: numbers are rounded so the color bar has about 100 steps, since the image can't show more
- legend_title is the text above the legend: "Value" on continuous maps, nothing on discrete maps
- data_type is "percentage" or "number": every MapQA title says which, and percentage ranges carry "%"
- train/dev/test follow MapQA's own question files; the few maps that have no questions are skipped

Usage:
    python mapqa/convert_mapqa.py --mapqa ~/Downloads/MapQA_S.zip --out data/mapqa_derender
    python mapqa/convert_mapqa.py --mapqa /kaggle/input/<your-dataset> --out /kaggle/working/mapqa_derender

--mapqa can be the zip or any folder that contains MapQA_S/.

Output: <out>/train.jsonl, dev.jsonl, test.jsonl with one line per map:
    {"id": "map_10", "image": "images/map_10.png", "target": "<the JSON above, as a string>",
     "legend_type": "discrete", "data_type": "number", "true_values": {"AK": 7603168, ...}}
"image" is relative to the MapQA_S folder. true_values are the exact numbers, for evaluation only.
"""

import argparse
import json
import math
import re
import statistics
import zipfile
from pathlib import Path

SPLITS = ("train", "dev", "test")
CONTINUOUS_LEGEND_TITLE = "Value"  # checked on several MapQA-S images; discrete maps show no legend title


class MapQASource:
    """Reads MapQA_S files from the downloaded zip or from an extracted folder."""

    def __init__(self, path):
        path = Path(path).expanduser()
        if path.is_file() and zipfile.is_zipfile(path):
            self.zip = zipfile.ZipFile(path)
            names = [n for n in self.zip.namelist() if not n.startswith("__MACOSX")]
            first_meta = next((n for n in names if re.search(r"(^|/)metadata/map_\d+\.json$", n)), None)
            if first_meta is None:
                raise FileNotFoundError(f"no metadata/map_*.json inside {path}")
            self.prefix = first_meta[: first_meta.index("metadata/")]
            self.names = {n[len(self.prefix):] for n in names if n.startswith(self.prefix)}
        else:
            self.zip = None
            patterns = ["*/" * depth + "metadata" for depth in range(5)]
            root = next((p.parent for pattern in patterns for p in path.glob(pattern)
                         if p.is_dir() and "__MACOSX" not in p.parts), None)
            if root is None:
                raise FileNotFoundError(f"no metadata/ folder found under {path}")
            self.root = root
        self.location = path

    def read(self, rel):
        if self.zip:
            return self.zip.read(self.prefix + rel)
        return (self.root / rel).read_bytes()

    def exists(self, rel):
        return rel in self.names if self.zip else (self.root / rel).exists()

    def metadata_ids(self):
        if self.zip:
            found = (re.fullmatch(r"metadata/map_(\d+)\.json", n) for n in self.names)
        else:
            found = (re.fullmatch(r"map_(\d+)\.json", p.name) for p in (self.root / "metadata").iterdir())
        return sorted(int(m.group(1)) for m in found if m)


def split_of_each_map(source):
    """map id -> split, from the map ids referenced in MapQA's question files."""
    split_of = {}
    for split in SPLITS:
        raw = source.read(f"questions/{split}-QA.json")
        for map_id in re.findall(rb'"map_id"\s*:\s*"map_(\d+)\.png"', raw):
            split_of[int(map_id)] = split
    return split_of


def is_missing(value):
    return value is None or (isinstance(value, float) and math.isnan(value))


def round_for_color_bar(values):
    """Round so the color bar spans about 100 steps: one decimal for a span of ~10, hundreds for ~10,000."""
    present = [v for v in values if v is not None]
    span = max(present) - min(present) if present else 0
    if span <= 0:
        return values
    digits = round(-math.log10(span / 100))
    if digits > 0:
        return [None if v is None else round(v, digits) for v in values]  # never adds digits to the data
    return [None if v is None else int(round(v, digits)) for v in values]


def convert_map(meta):
    """Return (target dict, true values) for one MapQA metadata entry."""
    map_id = meta["map_id"]
    data = sorted(meta["data"], key=lambda d: d["region"])
    if len(data) != 50:
        raise ValueError(f"map {map_id}: expected 50 states, got {len(data)}")
    codes = [d["region"] for d in data]
    true_values = {c: None if is_missing(d["value"]) else d["value"] for c, d in zip(codes, data)}
    title = meta["title"]["text"]
    data_type = meta["data_type"].lower()
    if data_type not in ("percentage", "number"):
        raise ValueError(f"map {map_id}: unknown data_type {meta['data_type']!r}")

    if meta["legend_type"] == "Continuous":
        values = round_for_color_bar([true_values[c] for c in codes])
        target = {"title": title, "legend_title": CONTINUOUS_LEGEND_TITLE, "legend_type": "continuous",
                  "data_type": data_type, "values": dict(zip(codes, values))}
    elif meta["legend_type"] == "Discrete":
        ranges = [c for c in meta["class_descriptions"] if c != "N/A"]
        classes = []
        for d in data:
            if d["class"] < 0:
                if d["description"] != "N/A":
                    raise ValueError(f"map {map_id}: {d['region']} has class {d['class']} but {d['description']!r}")
                classes.append(None)
            else:
                if ranges[d["class"]] != d["description"]:
                    raise ValueError(f"map {map_id}: {d['region']} class {d['class']} != {d['description']!r}")
                classes.append(d["class"])
        target = {"title": title, "legend_title": None, "legend_type": "discrete", "data_type": data_type,
                  "ranges": ranges, "values": dict(zip(codes, classes))}
    else:
        raise ValueError(f"map {map_id}: unknown legend_type {meta['legend_type']!r}")
    return target, true_values


def main():
    parser = argparse.ArgumentParser(description=__doc__, formatter_class=argparse.RawDescriptionHelpFormatter)
    parser.add_argument("--mapqa", required=True, help="MapQA_S.zip, or a folder containing MapQA_S/")
    parser.add_argument("--out", type=Path, default=Path("data/mapqa_derender"))
    parser.add_argument("--limit", type=int, help="convert only the first N maps (quick test)")
    args = parser.parse_args()

    source = MapQASource(args.mapqa)
    split_of = split_of_each_map(source)
    ids = source.metadata_ids()[: args.limit]
    args.out.mkdir(parents=True, exist_ok=True)

    files = {split: open(args.out / f"{split}.jsonl", "w", encoding="utf-8") for split in SPLITS}
    counts = {split: {"continuous": 0, "discrete": 0} for split in SPLITS}
    no_split, no_image, lengths = [], [], []
    try:
        for map_id in ids:
            split = split_of.get(map_id)
            if split is None:
                no_split.append(map_id)
                continue
            image = f"images/map_{map_id}.png"
            if not source.exists(image):
                no_image.append(map_id)
                continue
            meta = json.loads(source.read(f"metadata/map_{map_id}.json"))
            target, true_values = convert_map(meta)
            target_text = json.dumps(target, ensure_ascii=False, separators=(",", ":"))
            record = {
                "id": f"map_{map_id}",
                "image": image,
                "target": target_text,
                "legend_type": target["legend_type"],
                "data_type": target["data_type"],
                "true_values": true_values,
            }
            files[split].write(json.dumps(record, ensure_ascii=False) + "\n")
            counts[split][target["legend_type"]] += 1
            lengths.append(len(target_text))
    finally:
        for f in files.values():
            f.close()

    print(f"Read {len(ids)} maps from {source.location}")
    for split in SPLITS:
        c = counts[split]
        print(f"  {split:5} {c['continuous'] + c['discrete']:6} maps "
              f"({c['discrete']} discrete, {c['continuous']} continuous) -> {args.out / (split + '.jsonl')}")
    if no_split:
        print(f"  skipped {len(no_split)} map(s) that no question file uses: {no_split[:10]}")
    if no_image:
        print(f"  skipped {len(no_image)} map(s) with no image: {no_image[:10]}")
    if lengths:
        print(f"  target length in characters: min {min(lengths)}, median {int(statistics.median(lengths))}, "
              f"max {max(lengths)}")


if __name__ == "__main__":
    main()


In [ ]:
%%writefile common.py
"""Shared pieces of the derenderer training scripts: loading maps, the training loop, and scoring.

Training data is MapQA (from convert_mapqa.py) plus, with --synth, synthetic maps from
synth/generate_maps.py. Both use the same target JSON.

Scoring (per legend type; MapQA and synthetic test maps are scored separately):
    parsed                 output is a JSON object
    exact_match            output equals the target exactly
    title_correct          title matches (ignoring case and spacing)
    legend_title_correct   legend title matches (ignoring case and spacing; both null also counts)
    legend_type_correct    "continuous" / "discrete" is right
    data_type_correct      "percentage" / "number" is right
    coverage               share of the 50 states present in "values"
    extra_keys             keys in "values" that are not states of this map
  discrete maps:
    ranges_correct         the legend ranges match exactly
    state_accuracy         share of states with the right range index (null for no data)
  continuous maps (errors are relative to the true value range, so 0.05 = 5% of the range):
    nmae, nrmse            mean / root-mean-square error; a missing or non-numeric state counts as 1,
                           and no single error counts more than 1
    within_5pct            share of states within 5% of the range
Maps whose output doesn't parse count as 0 for accuracies and 1 for errors.
"""

import contextlib
import json
import math
import random
import time
from pathlib import Path

import torch
from PIL import Image

def add_common_args(parser):
    parser.add_argument("--data", type=Path, required=True, help="folder with MapQA train.jsonl and test.jsonl")
    parser.add_argument("--mapqa", type=Path, required=True, help="folder containing MapQA_S/images (searched a few levels)")
    parser.add_argument("--synth", type=Path, help="folder containing the synthetic maps (train.jsonl next to "
                                                   "images/synth_*, searched a few levels); adds them to training "
                                                   "and scoring")
    parser.add_argument("--out", type=Path, required=True)
    parser.add_argument("--train-maps", type=int, help="train on a random subset of this many MapQA maps")
    parser.add_argument("--synth-train-maps", type=int, help="train on a random subset of this many synthetic maps")
    parser.add_argument("--eval-maps", type=int, default=400, help="MapQA test maps to score (same maps for every model)")
    parser.add_argument("--synth-eval-maps", type=int, default=200, help="synthetic test maps to score")
    parser.add_argument("--eval-limit", type=int, help="score only the first N sampled test maps; a prefix of the "
                                                       "usual set, so results stay comparable with earlier runs")
    parser.add_argument("--epochs", type=int, default=2)
    parser.add_argument("--hours", type=float, default=8.0,
                        help="training time budget; the learning-rate schedule is shortened to fit it")
    parser.add_argument("--batch-size", type=int, default=4)
    parser.add_argument("--grad-accum", type=int, default=4)
    parser.add_argument("--lr", type=float, default=5e-5)
    parser.add_argument("--weight-decay", type=float, default=0.01)
    parser.add_argument("--eval-batch-size", type=int, default=8)
    parser.add_argument("--max-new-tokens", type=int, default=640, help="longest MapQA target is about 530 tokens")
    parser.add_argument("--workers", type=int, default=2)
    parser.add_argument("--log-every", type=int, default=20)
    parser.add_argument("--skip-train", action="store_true", help="only score the model as loaded")
    parser.add_argument("--seed", type=int, default=0)


def set_seed(seed):
    random.seed(seed)
    torch.manual_seed(seed)


def find_images_root(path, pattern):
    """The folder whose images/ holds files matching pattern, searched a few levels under path.

    Finding datasets by their files rather than folder names works however Kaggle mounts or unpacks them.
    """
    for depth in range(5):
        hits = Path(path).glob("*/" * depth + "images/" + pattern)
        hit = next((p for p in hits if "__MACOSX" not in p.parts), None)
        if hit:
            return hit.parent.parent
    raise FileNotFoundError(f"no images/{pattern} under {path}")


def find_mapqa_root(path):
    return find_images_root(path, "map_*.png")


def find_saved(path, marker):
    """A saved model folder: path itself when it holds marker, else a folder a few levels down that does.

    Lets a previous Kaggle run's output be passed as /kaggle/input. A name that is not a path, such as a
    Hugging Face model id, comes back unchanged.
    """
    path = Path(path)
    if not path.exists():
        return str(path)
    if (path / marker).exists():
        return str(path)
    for depth in range(1, 6):
        hit = next(path.glob("*/" * depth + marker), None)
        if hit:
            return str(hit.parent)
    raise FileNotFoundError(f"no {marker} under {path}")


def read_split(data_dir, split, image_root):
    """Records of one split, each with image_path set (a record's "image" is relative to image_root)."""
    records = []
    with open(Path(data_dir) / f"{split}.jsonl", encoding="utf-8") as f:
        for line in f:
            record = json.loads(line)
            record["image_path"] = str(Path(image_root) / record["image"])
            records.append(record)
    return records


def sample_records(records, n, seed):
    if n is None or n >= len(records):
        return list(records)
    return random.Random(seed).sample(records, n)


def training_records(args, mapqa_root):
    records = sample_records(read_split(args.data, "train", mapqa_root), args.train_maps, args.seed)
    synthetic = []
    if args.synth:
        folder = find_images_root(args.synth, "synth_*")
        synthetic = sample_records(read_split(folder, "train", folder), args.synth_train_maps, args.seed)
    print(f"training maps: {len(records)} MapQA + {len(synthetic)} synthetic", flush=True)
    return records + synthetic


def evaluation_sets(args, mapqa_root):
    """(name, records) pairs: MapQA test maps, plus synthetic test maps when --synth is given."""
    sets = [("mapqa", sample_records(read_split(args.data, "test", mapqa_root), args.eval_maps, seed=0))]
    if args.synth:
        folder = find_images_root(args.synth, "synth_*")
        sets.append(("synth", sample_records(read_split(folder, "test", folder), args.synth_eval_maps, seed=0)))
    if args.eval_limit:
        sets = [(name, records[: args.eval_limit]) for name, records in sets]
    return sets


class MapDataset(torch.utils.data.Dataset):
    def __init__(self, records):
        self.records = records

    def __len__(self):
        return len(self.records)

    def __getitem__(self, i):
        record = self.records[i]
        return Image.open(record["image_path"]).convert("RGB"), record["target"]


def as_list(batch):
    return batch


def autocast(device, amp_dtype):
    return torch.autocast(device.type, dtype=amp_dtype) if amp_dtype else contextlib.nullcontext()


def train_loop(model, loader, args, forward, out_dir, amp_dtype=None):
    """AdamW with warmup + cosine decay. Stops after --epochs or --hours, whichever comes first.

    forward(batch) returns the loss. After 20 updates the schedule is shortened, if needed, so the
    learning rate still decays to zero within the time budget.
    """
    params = [p for p in model.parameters() if p.requires_grad]
    optimizer = torch.optim.AdamW(params, lr=args.lr, weight_decay=args.weight_decay)
    schedule = {"total": max(1, args.epochs * math.ceil(len(loader) / args.grad_accum))}

    def lr_factor(update):
        warmup = max(1, round(0.03 * schedule["total"]))
        if update < warmup:
            return (update + 1) / warmup
        progress = min(1.0, (update - warmup) / max(1, schedule["total"] - warmup))
        return 0.5 * (1 + math.cos(math.pi * progress))

    scheduler = torch.optim.lr_scheduler.LambdaLR(optimizer, lr_factor)
    scaler = torch.amp.GradScaler("cuda", enabled=amp_dtype == torch.float16)
    device = next(model.parameters()).device
    start = time.time()
    deadline = start + args.hours * 3600
    update, update_loss, window = 0, 0.0, []
    model.train()
    with open(out_dir / "train_log.jsonl", "a") as log:
        for epoch in range(args.epochs):
            for i, batch in enumerate(loader, 1):
                with autocast(device, amp_dtype):
                    loss = forward(batch)
                scaler.scale(loss / args.grad_accum).backward()
                update_loss += loss.item() / args.grad_accum
                if i % args.grad_accum and i != len(loader):
                    continue
                scaler.unscale_(optimizer)
                torch.nn.utils.clip_grad_norm_(params, 1.0)
                scaler.step(optimizer)
                scaler.update()
                optimizer.zero_grad(set_to_none=True)
                scheduler.step()
                update += 1
                window.append(update_loss)
                update_loss = 0.0
                if update == 20:
                    seconds_per_update = (time.time() - start) / update
                    fits = update + int((deadline - time.time()) / seconds_per_update)
                    schedule["total"] = max(update + 1, min(schedule["total"], fits))
                if update == 1 or update % args.log_every == 0:
                    entry = {"epoch": epoch, "update": update, "of": schedule["total"],
                             "loss": round(sum(window) / len(window), 4), "lr": scheduler.get_last_lr()[0],
                             "minutes": round((time.time() - start) / 60, 1)}
                    window.clear()
                    print(json.dumps(entry), flush=True)
                    log.write(json.dumps(entry) + "\n")
                    log.flush()
                if update >= schedule["total"] or time.time() > deadline:
                    print(f"stopping after {update} updates ({(time.time() - start) / 3600:.2f} h)", flush=True)
                    return update
    return update


# ---------------------------------------------------------------- scoring

def parse_prediction(text):
    """The first JSON object in the model output, or None."""
    start = text.find("{")
    if start < 0:
        return None
    try:
        obj, _ = json.JSONDecoder().raw_decode(text[start:])
    except json.JSONDecodeError:
        return None
    return obj if isinstance(obj, dict) else None


def _same_text(a, b):
    if isinstance(a, str) and isinstance(b, str):
        return " ".join(a.split()).lower() == " ".join(b.split()).lower()
    return a is None and b is None


def _is_number(v):
    return isinstance(v, (int, float)) and not isinstance(v, bool) and math.isfinite(v)


def score_map(record, prediction_text):
    target = json.loads(record["target"])
    pred = parse_prediction(prediction_text)
    score = {"id": record["id"], "legend_type": target["legend_type"], "parsed": pred is not None}
    if pred is None:
        return score
    values = pred.get("values") if isinstance(pred.get("values"), dict) else {}
    codes = list(target["values"])
    score["exact_match"] = pred == target
    score["title_correct"] = _same_text(pred.get("title"), target["title"])
    score["legend_title_correct"] = _same_text(pred.get("legend_title"), target["legend_title"])
    score["legend_type_correct"] = pred.get("legend_type") == target["legend_type"]
    score["data_type_correct"] = pred.get("data_type") == target["data_type"]
    score["coverage"] = sum(c in values for c in codes) / len(codes)
    score["extra_keys"] = sum(k not in target["values"] for k in values)
    if target["legend_type"] == "discrete":
        score["ranges_correct"] = pred.get("ranges") == target["ranges"]
        right = {c: bool(c in values and values[c] == target["values"][c]) for c in codes}
        score["state_accuracy"] = sum(right.values()) / len(codes)
        score["per_state"] = right
    else:
        truth = {c: v for c, v in record["true_values"].items() if v is not None}
        span = (max(truth.values()) - min(truth.values())) or 1.0
        per_state = {c: min(1.0, abs(values[c] - v) / span) if _is_number(values.get(c)) else 1.0
                     for c, v in truth.items()}
        errors = list(per_state.values())
        score["nmae"] = sum(errors) / len(errors)
        score["nrmse"] = math.sqrt(sum(e * e for e in errors) / len(errors))
        score["within_5pct"] = sum(e <= 0.05 for e in errors) / len(errors)
        score["per_state"] = per_state
    return score


def summarize(scores):
    summary = {}
    for kind in ("discrete", "continuous"):
        group = [s for s in scores if s["legend_type"] == kind]
        if not group:
            continue

        def mean(key, if_unparsed):
            return round(sum(float(s.get(key, if_unparsed)) for s in group) / len(group), 4)

        result = {"maps": len(group)}
        for key in ("parsed", "exact_match", "title_correct", "legend_title_correct", "legend_type_correct",
                    "data_type_correct", "coverage"):
            result[key] = mean(key, 0)
        result["extra_keys"] = mean("extra_keys", 0)
        if kind == "discrete":
            result.update(ranges_correct=mean("ranges_correct", 0), state_accuracy=mean("state_accuracy", 0))
        else:
            result.update(nmae=mean("nmae", 1), nrmse=mean("nrmse", 1), within_5pct=mean("within_5pct", 0))
        result["per_state"] = per_state_summary(group, unparsed=0.0 if kind == "discrete" else 1.0)
        summary[kind] = result
    return summary


def per_state_summary(group, unparsed):
    """Per-state accuracy (discrete) or mean error (continuous), worst state first.

    Small states are the known weak point, so the run has to report them separately rather than
    leaving them averaged into the 50.
    """
    every_code = {code for score in group for code in score.get("per_state", {})}
    totals = {}
    for score in group:
        per_state = score.get("per_state") or {}
        for code in every_code:
            got = float(per_state[code]) if code in per_state else unparsed
            hit, count = totals.get(code, (0.0, 0))
            totals[code] = (hit + got, count + 1)
    means = {code: round(total / count, 4) for code, (total, count) in totals.items()}
    worst = sorted(means, key=lambda c: means[c], reverse=unparsed == 1.0)
    return {code: means[code] for code in worst}


def evaluate_and_save(records, predictions, out_dir, name):
    """Score one test set; writes predictions_<name>.jsonl and metrics_<name>.json."""
    scores = [score_map(r, p) for r, p in zip(records, predictions)]
    with open(out_dir / f"predictions_{name}.jsonl", "w", encoding="utf-8") as f:
        for record, prediction, score in zip(records, predictions, scores):
            f.write(json.dumps({"id": record["id"], "prediction": prediction, "target": record["target"],
                                "score": score}, ensure_ascii=False) + "\n")
    summary = summarize(scores)
    (out_dir / f"metrics_{name}.json").write_text(json.dumps(summary, indent=2))
    print(f"{name} test maps:\n{json.dumps(summary, indent=2)}", flush=True)
    return summary


In [ ]:
%%writefile tiles.py
#!/usr/bin/env python3
"""Choose where to magnify a map, from the picture alone.

Small regions are where the derenderer fails: a state a few pixels wide is smaller than one 28x28-pixel
vision token, so its colour is averaged with its neighbours before the model sees it. Rather than naming
the crowded corner by hand (which would only ever be right for one country), this picks the tiles with the
densest region boundaries and lets the model magnify those.

    boxes = busiest_tiles(image, tiles=2)     # boxes in the image's own pixels, biggest crowding first

The legend is full of edges too, so boundaries are only counted inside the map body: the largest connected
patch of non-background pixels, which is the mainland because its states all touch.
"""

import numpy as np
from scipy import ndimage

TILE_FRACTION = 0.16          # a tile covers this share of the map body's width
STRIDE_FRACTION = 0.5         # windows step by half a tile
TOKEN_PIXELS = 28 * 28        # one vision token covers a 28x28-pixel square of the map
SPECK_PIXELS = 40             # smaller than this is a rendering speck between borders, not a region


def map_body(image):
    """Mask of the mainland: the largest connected patch that is not the background colour."""
    pixels = np.asarray(image.convert("RGB"), dtype=np.int16)
    corners = np.stack([pixels[0, 0], pixels[0, -1], pixels[-1, 0], pixels[-1, -1]])
    background = np.median(corners, axis=0)
    filled = np.abs(pixels - background).max(axis=2) > 24
    labels, count = ndimage.label(filled)
    if count == 0:
        return np.ones(filled.shape, dtype=bool)
    sizes = ndimage.sum_labels(filled, labels, index=range(1, count + 1))
    return labels == (int(np.argmax(sizes)) + 1)


def boundary_mask(image, body):
    """Where region borders run: colour changes inside the map body."""
    grey = np.asarray(image.convert("L"), dtype=np.float32)
    steps = np.zeros_like(grey)
    steps[:-1, :-1] = np.abs(np.diff(grey, axis=0)[:, :-1]) + np.abs(np.diff(grey, axis=1)[:-1, :])
    return (steps > 20) & body


def small_regions(image, body, boundaries):
    """Centres of the regions that fit inside a single vision token, where this model loses colours.

    Regions are the patches left when the boundaries are removed. What matters is not how many borders a
    place has but how many of its regions are too small for the tower to resolve, so those are counted
    directly. Anything under a few dozen pixels is a speck where two borders meet, not a region.
    The pixel thresholds assume the map reaches the tower near its own resolution, which holds for maps
    of up to about a megapixel.
    """
    labels, count = ndimage.label(body & ~boundaries)
    if count == 0:
        return np.zeros((0, 2)), np.zeros(0)
    index = range(1, count + 1)
    areas = np.asarray(ndimage.sum_labels(body, labels, index=index))
    keep = (areas >= SPECK_PIXELS) & (areas < TOKEN_PIXELS)
    if not keep.any():
        return np.zeros((0, 2)), np.zeros(0)
    centres = np.asarray(ndimage.center_of_mass(body, labels, index=[i for i, k in zip(index, keep) if k]))
    return centres.reshape(-1, 2), areas[keep]


def busiest_tiles(image, tiles=2, tile_fraction=TILE_FRACTION):
    """Boxes over the parts of the map holding the most small regions, richest first, non-overlapping.

    Ties, and maps whose regions are all large, fall back to plain boundary density.
    """
    body = map_body(image)
    rows, columns = np.where(body)
    if rows.size == 0:
        return []
    top, bottom, left, right = rows.min(), rows.max(), columns.min(), columns.max()
    size = max(28, int((right - left) * tile_fraction))
    stride = max(7, int(size * STRIDE_FRACTION))
    boundaries = boundary_mask(image, body)
    integral = boundaries.cumsum(0).cumsum(1)
    centres, _ = small_regions(image, body, boundaries)

    def density(y, x):
        y2, x2 = y + size, x + size
        return (integral[y2, x2] - integral[y, x2] - integral[y2, x] + integral[y, x])

    def score(y, x):
        inside = 0
        if len(centres):
            inside = int(((centres[:, 0] >= y) & (centres[:, 0] < y + size) &
                          (centres[:, 1] >= x) & (centres[:, 1] < x + size)).sum())
        return inside * 1000 + density(y, x) / max(1.0, integral[-1, -1]) * 100

    windows = [(score(y, x), x, y)
               for y in range(top, max(top + 1, bottom - size), stride)
               for x in range(left, max(left + 1, right - size), stride)
               if y + size < integral.shape[0] and x + size < integral.shape[1]]
    chosen = []
    for score, x, y in sorted(windows, reverse=True):
        if all(abs(x - bx) >= size * 0.6 or abs(y - by) >= size * 0.6 for bx, by, _, _ in chosen):
            chosen.append((x, y, x + size, y + size))
        if len(chosen) == tiles:
            break
    return chosen


In [ ]:
%%writefile train_qwen.py
#!/usr/bin/env python3
"""LoRA fine-tune Qwen2.5-VL (3B by default) to turn a choropleth map into the target JSON, then score it.

    python train_qwen.py --data /kaggle/working/data --mapqa /kaggle/input --out /kaggle/working/qwen
    python train_qwen.py ... --synth /kaggle/input            # also train and score on synthetic maps
    python train_qwen.py ... --skip-train                     # score the base model zero-shot, same prompt
    python train_qwen.py ... --adapter <out>/adapter          # continue training, or score a saved adapter

Writes <out>/adapter (LoRA weights), train_log.jsonl, and predictions_<set>.jsonl + metrics_<set>.json for
each test set (mapqa, plus synth when --synth is given).
"""

import argparse
import json
import math
import os
import random
import time
from pathlib import Path

# Set before torch initialises CUDA. Two images per example leave a lot of reserved-but-unused memory on a
# T4; letting the allocator grow its segments avoids failing on fragmentation alone.
os.environ.setdefault("PYTORCH_ALLOC_CONF", "expandable_segments:True")
os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")

import torch
from peft import LoraConfig, PeftModel, get_peft_model, set_peft_model_state_dict
from PIL import Image
from torch.utils.data import DataLoader
from transformers import AutoProcessor, Qwen2_5_VLForConditionalGeneration

from common import (add_common_args, autocast, evaluate_and_save, evaluation_sets, find_mapqa_root,
                    find_saved, set_seed, train_loop, training_records)
from tiles import busiest_tiles

PROMPT = (
    "Read this choropleth map and return only JSON with these keys: "
    '"title"; "legend_title" (null if there is none); "legend_type" ("continuous" or "discrete"); '
    '"data_type" ("percentage" or "number"); '
    '"ranges" (discrete legends only: the legend ranges, lowest first); '
    '"values" (all 50 US states by state code: a number for a continuous legend, '
    "the index of the state's range for a discrete legend, null for no data)."
)
CROP_NOTE = (" The second image is a magnified view of part of the same map; "
             "use it to read the small regions it contains.")
TILES_NOTE = (" The extra images are magnified views of the parts of the same map holding the smallest "
              "regions; use them to read the regions they contain. Each one covers this part of the map, "
              "as a fraction of its width and height:")


def tile_images(image, count, zoom, rng=None, jitter=0):
    """Magnified views of wherever this map's region boundaries are densest.

    The boxes come from the picture alone (train/tiles.py), so nothing here knows which country it is
    looking at - the same rule finds the north-east of a US map and the small states of any other.
    """
    views, boxes = [], []
    for left, top, right, bottom in busiest_tiles(image, tiles=count):
        if jitter and rng is not None:
            shift_x, shift_y = rng.randint(-jitter, jitter), rng.randint(-jitter, jitter)
            left, right = max(0, left + shift_x), min(image.width, right + shift_x)
            top, bottom = max(0, top + shift_y), min(image.height, bottom + shift_y)
        view = image.crop((left, top, right, bottom))
        views.append(view.resize((max(28, round(view.width * zoom)), max(28, round(view.height * zoom))),
                                 Image.BICUBIC))
        boxes.append((left / image.width, top / image.height, right / image.width, bottom / image.height))
    return views, boxes
# Matches the attention and MLP layers of the language model, and the MLP layers of the vision encoder.
LORA_TARGETS = ["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"]
# Adds the vision encoder's attention and its patch merger. Qwen2.5-VL fuses vision attention into "qkv"
# and "proj", which the names above do not match, so the vision tower's attention is otherwise never trained.
VISION_LORA_REGEX = (r"(.*\.(?:q_proj|k_proj|v_proj|o_proj|gate_proj|up_proj|down_proj|qkv))"
                     r"|(.*\.attn\.proj)|(.*visual\.merger\.mlp\.[02])")


def resized_size(width, height, factor, min_pixels, max_pixels):
    """The size Qwen's processor resizes an image to: each side a multiple of factor, area within bounds."""
    height_out = max(factor, round(height / factor) * factor)
    width_out = max(factor, round(width / factor) * factor)
    if height_out * width_out > max_pixels:
        shrink = math.sqrt(height * width / max_pixels)
        height_out = max(factor, math.floor(height / shrink / factor) * factor)
        width_out = max(factor, math.floor(width / shrink / factor) * factor)
    elif height_out * width_out < min_pixels:
        grow = math.sqrt(min_pixels / (height * width))
        height_out = math.ceil(height * grow / factor) * factor
        width_out = math.ceil(width * grow / factor) * factor
    return width_out, height_out


def two_scale_images(processor, images, scale):
    """Each image at the size the tower normally sees, and the same image `scale` times larger."""
    image_processor = processor.image_processor
    factor = image_processor.patch_size * image_processor.merge_size
    coarse, fine = [], []
    for image in images:
        width, height = resized_size(image.width, image.height, factor,
                                     image_processor.min_pixels, image_processor.max_pixels)
        coarse.append(image.resize((width, height), Image.BICUBIC))
        fine.append(image.resize((width * scale, height * scale), Image.BICUBIC))
    return coarse, fine


def pool_to_coarse(fine, fine_grid, coarse_grid, merge, scale):
    """Average the fine pass's tokens over the area of each coarse token, image by image."""
    pooled, at = [], 0
    for (frames, fine_h, fine_w), (_, coarse_h, coarse_w) in zip(fine_grid.tolist(), coarse_grid.tolist()):
        rows, columns = coarse_h // merge, coarse_w // merge
        count = frames * rows * scale * columns * scale
        block = fine[at: at + count].view(frames, rows, scale, columns, scale, -1)
        at += count
        pooled.append(block.mean(dim=(2, 4)).reshape(frames * rows * columns, -1))
    return torch.cat(pooled)


class S2Visual(torch.nn.Module):
    """The vision tower run at two scales and fused - "scaling on scales".

    The second pass sees the map enlarged, so a state covering a fraction of one 28x28-pixel token in the
    first pass covers several tokens here. Those fine tokens are averaged back onto the coarse grid and
    concatenated with it, so the language model still receives exactly as many tokens as before; each one
    now also carries what its own patch of map looks like close up.

    The fusion starts as a pass-through of the coarse pass (the fine half of its weights is zero), so a
    model trained without S2 behaves identically until training moves those weights.
    """

    def __init__(self, visual, scale):
        super().__init__()
        self.visual = visual
        self.scale = scale
        width = visual.config.out_hidden_size
        self.fuse = torch.nn.Linear(2 * width, width, bias=False)
        with torch.no_grad():
            self.fuse.weight.zero_()
            self.fuse.weight[:, :width].copy_(torch.eye(width))
        self.pending = None

    def __getattr__(self, name):  # dtype, spatial_merge_size, config and friends come from the tower
        try:
            return super().__getattr__(name)
        except AttributeError:
            return getattr(self._modules["visual"], name)

    def forward(self, hidden_states, grid_thw, **kwargs):
        coarse = self.visual(hidden_states, grid_thw=grid_thw, **kwargs)
        pending, self.pending = self.pending, None
        if pending is None:
            return coarse
        pixels, fine_grid = pending
        with torch.no_grad():  # the tower is frozen here; only the fusion learns
            fine = self.visual(pixels.type(self.visual.dtype), grid_thw=fine_grid, **kwargs)
        pooled = pool_to_coarse(fine, fine_grid, grid_thw, self.visual.spatial_merge_size, self.scale)
        return self.fuse(torch.cat([coarse, pooled.to(coarse.dtype)], dim=-1))


def attach_s2(model, scale, device):
    """Replace the vision tower with the two-scale version, wherever it sits in the wrapped model."""
    for module in model.modules():
        if "visual" in module._modules and not isinstance(module._modules["visual"], S2Visual):
            wrapper = S2Visual(module._modules["visual"], scale)
            wrapper.fuse.to(device=device, dtype=torch.float32).requires_grad_(True)
            module._modules["visual"] = wrapper
            return wrapper
    raise SystemExit("could not find the vision tower to attach S2 to")

def fusion_tiles(image, processor, count, cells, zoom):
    """The busiest tiles, snapped to whole cells of the coarse token grid and cut out at `zoom` times size.

    Each tile covers exactly cells x cells coarse tokens, so its zoom x zoom fine tokens per cell line up
    with the coarse grid without any resampling: the tile's tokens can be added to the coarse tokens that
    already sit at those map positions.
    """
    reader = processor.image_processor
    factor = reader.patch_size * reader.merge_size
    width, height = resized_size(image.width, image.height, factor, reader.min_pixels, reader.max_pixels)
    columns, rows = width // factor, height // factor
    scale_x, scale_y = width / image.width, height / image.height
    placed = []
    for left, top, right, bottom in busiest_tiles(image, tiles=count):
        first_column = min(max(0, round((left + right) / 2 * scale_x / factor - cells / 2)), max(0, columns - cells))
        first_row = min(max(0, round((top + bottom) / 2 * scale_y / factor - cells / 2)), max(0, rows - cells))
        box = (first_column * factor / scale_x, first_row * factor / scale_y,
               (first_column + cells) * factor / scale_x, (first_row + cells) * factor / scale_y)
        side = cells * factor * zoom
        placed.append((first_row, first_column, image.resize((side, side), Image.BICUBIC, box=box)))
    return placed


def fusion_inputs(processor, maps, count, cells, zoom):
    """Pixels for every selected tile in a batch, and where each goes: (batch index, first row, first column)."""
    tiles, where = [], []
    for index, image in maps:
        for row, column, tile in fusion_tiles(image, processor, count, cells, zoom):
            tiles.append(tile)
            where.append((index, row, column))
    if not tiles:
        return {}
    seen = processor.image_processor(images=tiles, return_tensors="pt")
    return {"fx_pixel_values": seen["pixel_values"], "fx_grid_thw": seen["image_grid_thw"],
            "fx_where": torch.tensor(where)}


class FusionVisual(torch.nn.Module):
    """The vision tower, plus close-up detail added to the coarse tokens that sit on the busiest tiles.

    A tile is cut from the map, enlarged `zoom` times and run through the same frozen tower. Its tokens
    come back on a grid `zoom` times finer than the coarse one, so each coarse token has zoom x zoom fine
    tokens over exactly the same piece of map. Those are not averaged (averaging is how a state covering
    one fine token gets diluted to a ninth): each is shrunk, the block is laid side by side, and one
    linear layer maps the block to a correction that is added to the coarse token.

    Every other coarse token is untouched, and the language model still receives exactly the tokens it did
    before. The output layer starts at zero, so an untrained fusion changes nothing.
    """

    def __init__(self, visual, cells, zoom, reduced=256):
        super().__init__()
        self.visual, self.cells, self.zoom = visual, cells, zoom
        width = visual.config.out_hidden_size
        self.reduce = torch.nn.Linear(width, reduced)
        self.out = torch.nn.Linear(zoom * zoom * reduced, width, bias=False)
        with torch.no_grad():
            self.out.weight.zero_()
        self.pending = None

    def __getattr__(self, name):  # dtype, spatial_merge_size, config and friends come from the tower
        try:
            return super().__getattr__(name)
        except AttributeError:
            return getattr(self._modules["visual"], name)

    def forward(self, hidden_states, grid_thw, **kwargs):
        coarse = self.visual(hidden_states, grid_thw=grid_thw, **kwargs)
        pending, self.pending = self.pending, None
        if pending is None:
            return coarse
        pixels, tile_grid, where = pending
        with torch.no_grad():  # the tower is frozen here; only the fusion learns
            fine = self.visual(pixels.type(self.visual.dtype), grid_thw=tile_grid, **kwargs)
        cells, zoom = self.cells, self.zoom
        tiles = len(where)
        shrunk = self.reduce(fine.float()).view(tiles, cells, zoom, cells, zoom, -1)
        blocks = shrunk.permute(0, 1, 3, 2, 4, 5).reshape(tiles, cells * cells, -1)
        added = self.out(blocks)                                       # (tiles, cells * cells, width)
        merge = self.visual.spatial_merge_size
        rows_columns = [(h // merge, w // merge) for _, h, w in grid_thw.tolist()]
        offsets = [0]
        for rows, columns in rows_columns:
            offsets.append(offsets[-1] + rows * columns)
        delta = torch.zeros(coarse.shape, dtype=added.dtype, device=coarse.device)
        steps = torch.arange(cells, device=coarse.device)
        for k, (image, row, column) in enumerate(where.tolist()):
            index = offsets[image] + (row + steps)[:, None] * rows_columns[image][1] + (column + steps)[None, :]
            delta = delta.index_add(0, index.reshape(-1), added[k])
        return coarse + delta.to(coarse.dtype)


def attach_fusion(model, cells, zoom, device):
    """Replace the vision tower with the fusing version, wherever it sits in the wrapped model."""
    for module in model.modules():
        if "visual" in module._modules and not isinstance(module._modules["visual"], (S2Visual, FusionVisual)):
            wrapper = FusionVisual(module._modules["visual"], cells, zoom)
            wrapper.reduce.to(device=device, dtype=torch.float32).requires_grad_(True)
            wrapper.out.to(device=device, dtype=torch.float32).requires_grad_(True)
            module._modules["visual"] = wrapper
            return wrapper
    raise SystemExit("could not find the vision tower to attach the fusion to")


# Boxes in MapQA's 700x500 albers-usa layout; scaled if a map is a different size. NE_BOX was measured
# from the rendered maps: the small north-eastern fills span x 455-549, y 139-258 in every map.
NE_BOX = (440, 138, 565, 263)    # the small north-eastern states: RI, DE, CT, NJ, MA, NH, VT, MD
BODY_BOX = (100, 130, 560, 400)  # the map body, for crops that land elsewhere


def make_crop(image, rng, train):
    """A magnified crop of the map: biased to the north-east in training, fixed there at evaluation.

    Cropping alone would not help - the crop is enlarged so a state that covers a fraction of one
    28x28-pixel vision token in the full map covers several tokens here.
    """
    width, height = image.size
    scale_x, scale_y = width / 700, height / 500
    if train and rng.random() < 0.25:  # elsewhere in the body, so "image 2 zooms image 1" stays general
        left, top, right, bottom = BODY_BOX
        size = rng.uniform(110, 160)
        centre_x = rng.uniform(left + size / 2, right - size / 2)
        centre_y = rng.uniform(top + size / 2, bottom - size / 2)
        zoom = rng.uniform(2.5, 4.0)
    else:
        left, top, right, bottom = NE_BOX
        jitter = rng.uniform(-15, 15) if train else 0.0
        centre_x, centre_y = (left + right) / 2 + jitter, (top + bottom) / 2 + jitter
        size = rng.uniform(110, 165) if train else 130.0
        zoom = rng.uniform(2.5, 4.0) if train else 3.0
    half = size / 2
    box = (max(0, round((centre_x - half) * scale_x)), max(0, round((centre_y - half) * scale_y)),
           min(width, round((centre_x + half) * scale_x)), min(height, round((centre_y + half) * scale_y)))
    crop = image.crop(box)
    return crop.resize((max(28, round(crop.width * zoom)), max(28, round(crop.height * zoom))), Image.BICUBIC)


def tiles_note(boxes):
    """Say where each tile was cut from, so the model can place what it reads back on the map.

    Without this the extra images are just more pixels: the model can see that a region is dark but not
    which region it is, and has to match the tile to the map by eye before it can use it.
    """
    places = [f" view {i} x {left:.2f}-{right:.2f}, y {top:.2f}-{bottom:.2f}."
              for i, (left, top, right, bottom) in enumerate(boxes, start=2)]
    return TILES_NOTE + "".join(places)


def chat_text(processor, target=None, images=1, boxes=None):
    content = [{"type": "image"} for _ in range(images)]
    if boxes:
        note = tiles_note(boxes)
    else:
        note = "" if images == 1 else (CROP_NOTE if images == 2 else TILES_NOTE)
    content.append({"type": "text", "text": PROMPT + note})
    messages = [{"role": "user", "content": content}]
    if target is not None:
        messages.append({"role": "assistant", "content": [{"type": "text", "text": target}]})
    return processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=target is None)


class CropDataset(torch.utils.data.Dataset):
    """The full map, and on most examples a magnified crop as a second image.

    A share of examples keep the map alone (--crop-dropout), which keeps single-image inference in
    distribution and lets the crop be ablated at scoring time instead of in a second training run.
    """

    def __init__(self, records, dropout, tiles=0, tile_zoom=3.0, fuse=False):
        self.records, self.dropout = records, dropout
        self.tiles, self.tile_zoom, self.fuse = tiles, tile_zoom, fuse

    def __len__(self):
        return len(self.records)

    def __getitem__(self, i):
        record = self.records[i]
        image = Image.open(record["image_path"]).convert("RGB")
        boxes, fuse = [], False
        if random.random() < self.dropout:
            images = [image]                       # the map alone, so single-image use stays in distribution
        elif self.tiles and self.fuse:
            images, fuse = [image], True           # the tiles are added into the map's own tokens
        elif self.tiles:
            views, boxes = tile_images(image, self.tiles, self.tile_zoom, random, jitter=8)
            images = [image] + views
        else:
            images = [image, make_crop(image, random, train=True)]
        return images, boxes, record["target"], fuse


class TrainCollate:
    """Image(s) + prompt + answer, with labels only on the answer tokens."""

    def __init__(self, processor, fine_processor=None, scale=2, fusion=None):
        self.processor = processor
        self.fusion = fusion  # (tiles, cells, zoom) when tiles are fused into the map's tokens
        self.fine_processor = fine_processor  # same processor, allowed more pixels, for the second scale
        self.scale = scale
        self.answer_start = processor.tokenizer.encode("<|im_start|>assistant\n", add_special_tokens=False)

    def __call__(self, batch):
        texts, images = [], []
        fused = []
        for index, (sample_images, boxes, target, fuse) in enumerate(batch):
            if fuse:
                fused.append((index, sample_images[0]))
            images.extend(sample_images)
            texts.append(chat_text(self.processor, target, len(sample_images), boxes))
        fine = None
        if self.fine_processor is not None:
            images, fine_images = two_scale_images(self.processor, images, self.scale)
            fine = self.fine_processor.image_processor(images=fine_images, return_tensors="pt")
        enc = self.processor(text=texts, images=images, padding=True, return_tensors="pt")
        if fine is not None:
            enc["s2_pixel_values"], enc["s2_grid_thw"] = fine["pixel_values"], fine["image_grid_thw"]
        if fused:
            for key, value in fusion_inputs(self.processor, fused, *self.fusion).items():
                enc[key] = value
        labels = enc["input_ids"].clone()
        labels[enc["attention_mask"] == 0] = -100
        marker = torch.tensor(self.answer_start)
        for row, ids in enumerate(enc["input_ids"]):
            hits = (ids.unfold(0, len(marker), 1) == marker).all(dim=1).nonzero()
            labels[row, : int(hits[-1]) + len(marker)] = -100
        enc["labels"] = labels
        return enc


def predict(model, processor, records, args, device, amp_dtype, crop=False, s2=None, fine_processor=None,
            tiles=0, fusion=None):
    predictions = []
    began = time.time()
    with torch.no_grad():
        for start in range(0, len(records), args.eval_batch_size):
            batch = records[start : start + args.eval_batch_size]
            texts, images, fused = [], [], []
            for record in batch:
                image = Image.open(record["image_path"]).convert("RGB")
                images.append(image)
                extra, boxes = [], []
                if tiles and fusion is not None:
                    fused.append((len(texts), image))
                elif tiles:
                    extra, boxes = tile_images(image, tiles, args.tile_zoom)
                elif crop:
                    extra = [make_crop(image, random, train=False)]
                images.extend(extra)
                texts.append(chat_text(processor, images=1 + len(extra), boxes=boxes))
            fine = None
            if fine_processor is not None:
                images, fine_images = two_scale_images(processor, images, args.s2_scale)
                fine = fine_processor.image_processor(images=fine_images, return_tensors="pt")
            enc = processor(text=texts, images=images, padding=True, return_tensors="pt").to(device)
            if fused:
                seen = fusion_inputs(processor, fused, tiles, args.fusion_cells, args.fusion_zoom)
                fusion.pending = (seen["fx_pixel_values"].to(device), seen["fx_grid_thw"].to(device),
                                  seen["fx_where"].to(device))
            if fine is not None:
                s2.pending = (fine["pixel_values"].to(device), fine["image_grid_thw"].to(device))
            with autocast(device, amp_dtype):
                ids = model.generate(**enc, max_new_tokens=args.max_new_tokens, do_sample=False,
                                     repetition_penalty=1.0, use_cache=True)
            predictions += processor.batch_decode(ids[:, enc["input_ids"].shape[1]:], skip_special_tokens=True)
            print(f"predicted {len(predictions)}/{len(records)} ({(time.time() - began) / 60:.1f} min)", flush=True)
    return predictions


def report_lora(model):
    """Print where LoRA actually landed. PEFT matches module names, so a target that matches nothing is
    silently a no-op; this run's whole question is whether vision attention helps, so it must not pass quietly."""
    names = [name for name, _ in model.named_modules() if name.endswith("lora_A")]
    vision_attention = sum(1 for n in names if ".visual." in n and ".attn." in n)
    vision_mlp = sum(1 for n in names if ".visual." in n and ".mlp." in n and "merger" not in n)
    merger = sum(1 for n in names if "merger" in n)
    print(f"LoRA modules: {len(names)} total | vision attention {vision_attention} | "
          f"vision MLP {vision_mlp} | merger {merger} | language {len(names) - vision_attention - vision_mlp - merger}",
          flush=True)
    return vision_attention


def load_prior_adapter(model, path):
    """Load an earlier adapter into a model whose LoRA now covers more modules.

    LoRA initialises its B matrix to zero, so modules the earlier run did not train start as exact
    no-ops and the model begins exactly where that run ended.
    """
    from safetensors.torch import load_file

    state = load_file(str(Path(path) / "adapter_model.safetensors"))
    result = set_peft_model_state_dict(model, state)
    unexpected = list(getattr(result, "unexpected_keys", []))
    if unexpected:
        raise SystemExit(f"prior adapter does not line up: {len(unexpected)} unexpected keys, "
                         f"e.g. {unexpected[:3]}")
    print(f"loaded {len(state)} tensors from the prior adapter", flush=True)


def main():
    parser = argparse.ArgumentParser(description=__doc__, formatter_class=argparse.RawDescriptionHelpFormatter)
    add_common_args(parser)
    parser.add_argument("--model", default="Qwen/Qwen2.5-VL-3B-Instruct")
    parser.add_argument("--adapter", help="a saved LoRA adapter to continue from or to score")
    parser.add_argument("--lora-r", type=int, default=16)
    parser.add_argument("--max-pixels", type=int, default=640_000,
                        help="larger images are scaled down to this many pixels (MapQA maps have 350,000)")
    parser.add_argument("--vision-lora", action="store_true",
                        help="also train the vision encoder's attention and patch merger, not just its MLPs")
    parser.add_argument("--crop", action="store_true",
                        help="pass a magnified crop of the map as a second image")
    parser.add_argument("--crop-dropout", type=float, default=0.2,
                        help="share of training examples that keep the full map alone")
    parser.add_argument("--eval-both", action="store_true",
                        help="with --crop, also score without the crop, to separate its effect")
    parser.add_argument("--tiles", type=int, default=0,
                        help="magnify this many of the map's most crowded areas and pass them as extra "
                             "images; the areas are found from the picture, not from known geography")
    parser.add_argument("--tile-zoom", type=float, default=3.0, help="how much to magnify each tile")
    parser.add_argument("--fusion", action="store_true",
                        help="with --tiles: add each tile's close-up tokens into the map's own tokens instead of "
                             "passing the tile as an extra image, so the token count does not grow")
    parser.add_argument("--fusion-cells", type=int, default=3,
                        help="a fused tile covers this many coarse tokens on each side")
    parser.add_argument("--fusion-zoom", type=int, default=3,
                        help="whole-number magnification of a fused tile (fine tokens per coarse token, per side)")
    parser.add_argument("--s2", action="store_true",
                        help="run the vision tower at two scales and fuse them, instead of cropping by hand")
    parser.add_argument("--s2-scale", type=int, default=2,
                        help="how many times larger the second pass sees the map")
    # A T4 trains about 2.3 s per MapQA map; 8 h leaves room for scoring inside Kaggle's 12-hour limit
    parser.set_defaults(epochs=1, hours=8.0, lr=2e-4, batch_size=2, grad_accum=8, eval_batch_size=16,
                        max_new_tokens=600)
    args = parser.parse_args()
    set_seed(args.seed)
    args.out.mkdir(parents=True, exist_ok=True)
    (args.out / "args.json").write_text(json.dumps(vars(args), default=str, indent=1))

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    if device.type == "cuda":  # T4 and P100 have no bfloat16
        dtype = torch.bfloat16 if torch.cuda.get_device_capability()[0] >= 8 else torch.float16
    else:
        dtype = torch.float32
    amp_dtype = dtype if device.type == "cuda" else None
    root = find_mapqa_root(args.mapqa)
    processor = AutoProcessor.from_pretrained(args.model, max_pixels=args.max_pixels)
    processor.tokenizer.padding_side = "left"
    model = Qwen2_5_VLForConditionalGeneration.from_pretrained(args.model, torch_dtype=dtype,
                                                               attn_implementation="sdpa").to(device)
    targets = VISION_LORA_REGEX if args.vision_lora else LORA_TARGETS
    lora = LoraConfig(r=args.lora_r, lora_alpha=2 * args.lora_r, lora_dropout=0.05,
                      target_modules=targets, task_type="CAUSAL_LM")
    if args.adapter and args.vision_lora:
        # the saved adapter covers fewer modules than this config, so build the wider one and fill it in
        model = get_peft_model(model, lora)
        load_prior_adapter(model, find_saved(args.adapter, "adapter_config.json"))
    elif args.adapter:
        model = PeftModel.from_pretrained(model, find_saved(args.adapter, "adapter_config.json"),
                                          is_trainable=not args.skip_train)
    elif not args.skip_train:
        model = get_peft_model(model, lora)
    if args.adapter or not args.skip_train:
        vision_attention = report_lora(model)
        if args.vision_lora and not vision_attention:
            raise SystemExit("--vision-lora matched no vision attention; check VISION_LORA_REGEX")

    if args.fusion and (args.s2 or not args.tiles):
        raise SystemExit("--fusion needs --tiles N and cannot be combined with --s2")
    fusion = None
    if args.fusion:
        fusion = attach_fusion(model, args.fusion_cells, args.fusion_zoom, device)
        added = sum(p.numel() for p in (*fusion.reduce.parameters(), *fusion.out.parameters()))
        print(f"fusion: {args.tiles} tiles of {args.fusion_cells}x{args.fusion_cells} coarse tokens at "
              f"{args.fusion_zoom}x, {added:,} trainable weights", flush=True)

    s2 = fine_processor = None
    if args.s2:
        s2 = attach_s2(model, args.s2_scale, device)
        fine_processor = AutoProcessor.from_pretrained(args.model,
                                                       max_pixels=args.max_pixels * args.s2_scale ** 2)
        print(f"S2: second pass at {args.s2_scale}x the size, fusing into "
              f"{s2.fuse.weight.numel():,} trainable weights", flush=True)

    if not args.skip_train:
        model.print_trainable_parameters()
        model.config.use_cache = False
        model.gradient_checkpointing_enable(gradient_checkpointing_kwargs={"use_reentrant": False})
        model.enable_input_require_grads()
        train = training_records(args, root)
        dataset = CropDataset(train, args.crop_dropout if (args.crop or args.tiles) else 1.0,
                              args.tiles, args.tile_zoom, fuse=args.fusion)
        collate = TrainCollate(processor, fine_processor, args.s2_scale,
                               (args.tiles, args.fusion_cells, args.fusion_zoom) if args.fusion else None)
        loader = DataLoader(dataset, batch_size=args.batch_size, shuffle=True, num_workers=args.workers,
                            collate_fn=collate)

        def forward(batch):
            batch = batch.to(device)
            if s2 is not None:
                s2.pending = (batch.pop("s2_pixel_values"), batch.pop("s2_grid_thw"))
            if fusion is not None:  # absent when every example in the batch kept the map alone
                pixels = batch.pop("fx_pixel_values", None)
                grid, where = batch.pop("fx_grid_thw", None), batch.pop("fx_where", None)
                fusion.pending = None if pixels is None else (pixels, grid, where)
            return model(**batch).loss

        print(f"{len(loader)} batches per epoch", flush=True)
        train_loop(model, loader, args, forward, args.out, amp_dtype)
        model.save_pretrained(args.out / "adapter")
        if s2 is not None:  # the fusion is not part of the LoRA adapter, so it is saved beside it
            torch.save(s2.fuse.state_dict(), args.out / "s2_fuse.pt")
        if fusion is not None:
            torch.save({"reduce": fusion.reduce.state_dict(), "out": fusion.out.state_dict()},
                       args.out / "fusion.pt")

    model.eval()
    if device.type == "cuda":
        torch.cuda.empty_cache()  # release memory held from training before generating
    variants = [("", args.crop or bool(args.tiles))]
    if (args.crop or args.tiles) and args.eval_both:
        variants.append(("_nocrop", False))  # same weights, extra views withheld: their effect on its own
    for name, records in evaluation_sets(args, root):
        for suffix, use_crop in variants:
            print(f"scoring {len(records)} {name} test maps{' without the crop' if suffix else ''}", flush=True)
            predictions = predict(model, processor, records, args, device, amp_dtype, crop=use_crop,
                                  s2=s2, fine_processor=fine_processor,
                                  tiles=args.tiles if use_crop else 0, fusion=fusion)
            evaluate_and_save(records, predictions, args.out, name + suffix)


if __name__ == "__main__":
    main()


In [ ]:
run("python convert_mapqa.py --mapqa /kaggle/input --out /kaggle/working/data")

In [ ]:
run('python train_qwen.py --data /kaggle/working/data --mapqa /kaggle/input --out /kaggle/working/qwen --adapter /kaggle/input --tiles 2 --fusion --eval-both --crop-dropout 0.2 --hours 3 --eval-limit 120 --batch-size 1 --grad-accum 16 --eval-batch-size 2 --log-every 10')